# 01 - Load Results

Load experiment data from JSONL/Parquet files and explore basic statistics.


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

ANALYSIS_ROOT = Path("..")
DATA_DIR = ANALYSIS_ROOT / "data"


In [ ]:
# List available experiments
if DATA_DIR.exists():
    experiments = [d for d in DATA_DIR.iterdir() if d.is_dir()]
    print(f"Found {len(experiments)} experiments:")
    for exp in sorted(experiments):
        merged_dir = exp / "merged"
        if merged_dir.exists():
            files = list(merged_dir.glob("*"))
            print(f"  • {exp.name} ({len(files)} files in merged/)")
        else:
            print(f"  • {exp.name} (no merged data yet)")
else:
    print("No data directory found. Run fetch_results.py first.")


In [ ]:
# Load experiment data
EXPERIMENT_ID = "exp_2025_01_01_001"  # Change this

def load_experiment(experiment_id: str) -> pd.DataFrame:
    exp_dir = DATA_DIR / experiment_id / "merged"
    parquet_path = exp_dir / "merged.parquet"
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    jsonl_path = exp_dir / "merged.jsonl"
    if jsonl_path.exists():
        return pd.read_json(jsonl_path, lines=True)
    raise FileNotFoundError(f"No data found for {experiment_id}")

try:
    df = load_experiment(EXPERIMENT_ID)
    print(f"Loaded {len(df):,} events")
except FileNotFoundError:
    # Create mock data for demonstration
    np.random.seed(42)
    n = 10000
    df = pd.DataFrame({
        "run_id": ["run_001"] * n,
        "event_id": range(n),
        "timestamp_utc_iso": pd.date_range("2025-01-01", periods=n, freq="100ms"),
        "operation": np.random.choice(["encrypt", "decrypt", "sign", "verify"], n),
        "algorithm": np.random.choice(["RSA-2048", "ECDSA-P256", "Kyber-768"], n),
        "latency_us": np.random.lognormal(6, 0.5, n).astype(int),
        "queue_delay_us": np.random.exponential(100, n).astype(int),
        "worker_id": np.random.randint(0, 4, n),
    })
    df["timestamp"] = pd.to_datetime(df["timestamp_utc_iso"])
    print(f"Created mock dataset with {len(df):,} events")


In [ ]:
# Data overview
print("Data shape:", df.shape)
print("\nColumns:")
print(df.dtypes)
print("\nPreview:")
df.head()


In [ ]:
# Basic statistics
if "latency_us" in df.columns:
    lat = df["latency_us"]
    print("Latency Statistics (μs):")
    print(f"  Mean:   {lat.mean():,.1f}")
    print(f"  Std:    {lat.std():,.1f}")
    print(f"  p50:    {lat.quantile(0.50):,.0f}")
    print(f"  p90:    {lat.quantile(0.90):,.0f}")
    print(f"  p99:    {lat.quantile(0.99):,.0f}")

# Store for other notebooks
%store df


# 01 - Load Results

Load experiment results from JSONL/Parquet files and preview basic statistics.


In [ ]:
# Configuration - modify these for your experiment
EXPERIMENT_ID = "exp_2025_0101_001"  # Change this
DATA_DIR = f"../data/{EXPERIMENT_ID}/merged"

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
def load_experiment_data(data_dir: str) -> pd.DataFrame:
    """Load experiment data from merged parquet or jsonl."""
    data_path = Path(data_dir)
    
    parquet_path = data_path / "merged.parquet"
    if parquet_path.exists():
        print(f"Loading from {parquet_path}")
        return pd.read_parquet(parquet_path)
    
    jsonl_path = data_path / "merged.jsonl"
    if jsonl_path.exists():
        print(f"Loading from {jsonl_path}")
        records = []
        with open(jsonl_path) as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))
        return pd.DataFrame(records)
    
    raise FileNotFoundError(f"No data found in {data_path}")

try:
    df = load_experiment_data(DATA_DIR)
    print(f"\n✓ Loaded {len(df):,} records")
except FileNotFoundError as e:
    print(f"✗ {e}")
    print(f"\nRun: python scripts/fetch_results.py -e {EXPERIMENT_ID} -u <uri> -o data/{EXPERIMENT_ID}")


In [ ]:
# Preview and basic stats
print("Schema:", df.dtypes.to_dict())
print(f"\nShape: {df.shape}")
df.head()


In [ ]:
# Quick visualization
if "latency_us" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(df["latency_us"], bins=100, edgecolor="white", alpha=0.7)
    axes[0].set_xlabel("Latency (μs)")
    axes[0].set_ylabel("Frequency")
    axes[0].set_title("Latency Distribution")
    
    if "timestamp_utc" in df.columns or "timestamp_utc_iso" in df.columns:
        ts_col = "timestamp_utc" if "timestamp_utc" in df.columns else "timestamp_utc_iso"
        df["_ts"] = pd.to_datetime(df[ts_col])
        throughput = df.set_index("_ts").resample("1S").size()
        axes[1].plot(throughput.values, alpha=0.7)
        axes[1].set_xlabel("Time (seconds)")
        axes[1].set_ylabel("Ops/second")
        axes[1].set_title("Throughput Over Time")
    
    plt.tight_layout()
    plt.show()
